In [1]:
# ═══════════════════════════════════════════════════
# SEMAINE 3 — Core AutoML : FLAML + Optuna + SHAP
# ═══════════════════════════════════════════════════
import pandas as pd
import numpy as np
import time
import warnings
import os
warnings.filterwarnings('ignore')

os.makedirs('../reports', exist_ok=True)

# Charger les datasets nettoyés
df_heart    = pd.read_csv('../datasets/processed/heart_disease_clean.csv')
df_diabetes = pd.read_csv('../datasets/processed/diabetes_clean.csv')

print("Datasets chargés ✓")
print(f"Heart Disease  : {df_heart.shape[0]} lignes × {df_heart.shape[1]} colonnes")
print(f"Diabetes       : {df_diabetes.shape[0]} lignes × {df_diabetes.shape[1]} colonnes")

# Résultat baseline à battre
BASELINE_F1  = 0.8814
BASELINE_ACC = 0.8852
print(f"\nBaseline à battre → F1 : {BASELINE_F1} | Accuracy : {BASELINE_ACC}")

Datasets chargés ✓
Heart Disease  : 303 lignes × 14 colonnes
Diabetes       : 768 lignes × 9 colonnes

Baseline à battre → F1 : 0.8814 | Accuracy : 0.8852


In [2]:
# ═══════════════════════════════════════════════════
# MODULE 1 : AutoPreprocessor (version corrigée)
# ═══════════════════════════════════════════════════
from sklearn.impute import SimpleImputer
from feature_engine.encoding import OrdinalEncoder
from feature_engine.outliers import Winsorizer
from sklearn.model_selection import train_test_split

class AutoPreprocessor:
    """
    Preprocessing automatique universel.
    Détecte les types de colonnes et applique les transformations adaptées.
    Ignore les colonnes binaires pour le traitement des outliers.
    """
    def __init__(self):
        self.num_cols        = []
        self.cat_cols        = []
        self.binary_cols     = []
        self.continuous_cols = []
        self.fitted          = False

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        # Détecter les types de colonnes
        self.num_cols = X.select_dtypes(include=['number']).columns.tolist()
        self.cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()

        # Séparer colonnes binaires et continues
        self.binary_cols     = [c for c in self.num_cols if X[c].nunique() <= 2]
        self.continuous_cols = [c for c in self.num_cols if X[c].nunique() > 2]

        # 1. Imputation des valeurs manquantes
        if self.num_cols:
            imputer = SimpleImputer(strategy='median')
            X[self.num_cols] = imputer.fit_transform(X[self.num_cols])

        # 2. Encodage des variables catégorielles
        if self.cat_cols:
            for col in self.cat_cols:
                X[col] = X[col].astype(str)
            encoder = OrdinalEncoder(variables=self.cat_cols)
            X = encoder.fit_transform(X)

        # 3. Traitement des outliers — uniquement sur colonnes continues
        if self.continuous_cols:
            winsorizer = Winsorizer(
                variables=self.continuous_cols,
                capping_method='iqr',
                tail='both', fold=1.5
            )
            X = winsorizer.fit_transform(X)

        self.fitted = True
        return X

    def get_info(self):
        return {
            "colonnes_numeriques"    : self.num_cols,
            "colonnes_binaires"      : self.binary_cols,
            "colonnes_continues"     : self.continuous_cols,
            "colonnes_categorielles" : self.cat_cols,
            "fitted"                 : self.fitted
        }

# ── Test sur Heart Disease
print("Test AutoPreprocessor sur Heart Disease...")
X_heart = df_heart.drop('target', axis=1)
y_heart = df_heart['target']

preprocessor = AutoPreprocessor()
X_processed  = preprocessor.fit_transform(X_heart)

print(f"\nAvant  — Valeurs manquantes : {X_heart.isnull().sum().sum()}")
print(f"Après  — Valeurs manquantes : {X_processed.isnull().sum().sum()}")
print(f"Shape  : {X_processed.shape}")
print(f"\nColonnes binaires  (pas d'outlier) : {preprocessor.binary_cols}")
print(f"Colonnes continues (outliers traités) : {preprocessor.continuous_cols}")
print("\nAutoPreprocessor OK ✓")

Test AutoPreprocessor sur Heart Disease...

Avant  — Valeurs manquantes : 6
Après  — Valeurs manquantes : 0
Shape  : (303, 13)

Colonnes binaires  (pas d'outlier) : ['sex', 'fbs', 'exang']
Colonnes continues (outliers traités) : ['age', 'cp', 'trestbps', 'chol', 'restecg', 'thalach', 'oldpeak', 'slope', 'ca', 'thal']

AutoPreprocessor OK ✓


In [3]:
# ═══════════════════════════════════════════════════
# MODULE 2 : AutoMLEngine avec FLAML
# ═══════════════════════════════════════════════════
from flaml import AutoML
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report

class AutoMLEngine:
    """
    Moteur AutoML basé sur FLAML.
    Explore automatiquement algorithmes et hyperparamètres
    dans un budget de temps défini.
    """
    def __init__(self, task='classification', time_budget=120):
        self.task        = task
        self.time_budget = time_budget
        self.automl      = AutoML()
        self.results     = {}
        self.trained     = False

    def fit(self, X_train, y_train):
        print(f"Lancement FLAML — budget : {self.time_budget}s ...")
        start = time.time()

        self.automl.fit(
            X_train, y_train,
            task        = self.task,
            time_budget = self.time_budget,
            metric      = 'f1',
            estimator_list = ['xgboost','lgbm','rf','catboost','extra_tree','lrl1'],
            seed        = 42,
            verbose     = 1
        )
        self.results['training_time']  = round(time.time() - start, 2)
        self.results['best_estimator'] = self.automl.best_estimator
        self.results['best_config']    = self.automl.best_config
        self.trained = True

        print(f"\nMeilleur modèle   : {self.automl.best_estimator}")
        print(f"Meilleure config  : {self.automl.best_config}")
        print(f"Temps total       : {self.results['training_time']}s")
        return self

    def evaluate(self, X_test, y_test):
        if not self.trained:
            raise Exception("Appelez fit() avant evaluate()")

        preds = self.automl.predict(X_test)
        self.results['accuracy'] = round(accuracy_score(y_test, preds), 4)
        self.results['f1_score'] = round(f1_score(y_test, preds, average='weighted'), 4)

        try:
            proba = self.automl.predict_proba(X_test)[:,1]
            self.results['auc_roc'] = round(roc_auc_score(y_test, proba), 4)
        except:
            self.results['auc_roc'] = None

        return self.results

print("AutoMLEngine défini ✓")
print("Prêt à entraîner avec FLAML !")

AutoMLEngine défini ✓
Prêt à entraîner avec FLAML !


In [4]:
# ═══════════════════════════════════════════════════
# EXPÉRIENCE 1 : FLAML sur Heart Disease
# ═══════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

print("=" * 55)
print("EXPÉRIENCE 1 — FLAML vs Baseline")
print("Dataset : Heart Disease | Budget : 120s")
print("=" * 55)

# Preprocessing automatique
preprocessor_h = AutoPreprocessor()
X_processed    = preprocessor_h.fit_transform(df_heart.drop('target', axis=1))
y_heart        = df_heart['target']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_heart,
    test_size=0.2, random_state=42, stratify=y_heart
)
print(f"\nTrain : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

# Lancement FLAML
engine = AutoMLEngine(task='classification', time_budget=120)
engine.fit(X_train, y_train)

# Évaluation
results = engine.evaluate(X_test, y_test)

# Comparaison avec baseline
print("\n" + "=" * 55)
print("RÉSULTATS COMPARATIFS")
print("=" * 55)
print(f"{'Méthode':<25} {'F1-Score':<12} {'Accuracy':<12} {'AUC-ROC'}")
print("-" * 55)
print(f"{'Baseline (RF défaut)':<25} {BASELINE_F1:<12} {BASELINE_ACC:<12} {'N/A'}")
print(f"{'FLAML AutoML':<25} {results['f1_score']:<12} {results['accuracy']:<12} {results['auc_roc']}")
print("-" * 55)

diff_f1 = round(results['f1_score'] - BASELINE_F1, 4)
print(f"\nAmélioration F1 : {'+' if diff_f1 > 0 else ''}{diff_f1}")
if diff_f1 > 0:
    print("FLAML bat la baseline ✓")
else:
    print("Baseline non battue — on va augmenter le budget")

EXPÉRIENCE 1 — FLAML vs Baseline
Dataset : Heart Disease | Budget : 120s

Train : 242 lignes | Test : 61 lignes
Lancement FLAML — budget : 120s ...

Meilleur modèle   : xgboost
Meilleure config  : {'n_estimators': 5, 'max_leaves': 18, 'min_child_weight': np.float64(5.551090747429498), 'learning_rate': np.float64(0.3134337917825165), 'subsample': np.float64(0.6842787685132097), 'colsample_bylevel': np.float64(0.8738716941536381), 'colsample_bytree': 1.0, 'reg_alpha': np.float64(0.017478427937785885), 'reg_lambda': np.float64(0.30775985534027894)}
Temps total       : 120.12s

RÉSULTATS COMPARATIFS
Méthode                   F1-Score     Accuracy     AUC-ROC
-------------------------------------------------------
Baseline (RF défaut)      0.8814       0.8852       N/A
FLAML AutoML              0.8526       0.8525       0.9367
-------------------------------------------------------

Amélioration F1 : -0.0288
Baseline non battue — on va augmenter le budget


In [5]:
# ═══════════════════════════════════════════════════
# EXPÉRIENCE 2 — FLAML budget 300s
# ═══════════════════════════════════════════════════
print("=" * 55)
print("EXPÉRIENCE 2 — FLAML budget étendu (300s)")
print("=" * 55)

engine_300 = AutoMLEngine(task='classification', time_budget=300)
engine_300.fit(X_train, y_train)
results_300 = engine_300.evaluate(X_test, y_test)

print("\n" + "=" * 55)
print("RÉSULTATS — 3 configurations comparées")
print("=" * 55)
print(f"{'Méthode':<28} {'F1':<10} {'Accuracy':<12} {'AUC-ROC':<10} {'Temps'}")
print("-" * 70)
print(f"{'Baseline RF défaut':<28} {BASELINE_F1:<10} {BASELINE_ACC:<12} {'N/A':<10} {'0.12s'}")
print(f"{'FLAML 120s (XGBoost)':<28} {results['f1_score']:<10} {results['accuracy']:<12} {results['auc_roc']:<10} {'120s'}")
print(f"{'FLAML 300s':<28} {results_300['f1_score']:<10} {results_300['accuracy']:<12} {results_300['auc_roc']:<10} {'300s'}")
print("-" * 70)

best_f1 = max(results['f1_score'], results_300['f1_score'])
diff    = round(best_f1 - BASELINE_F1, 4)
print(f"\nMeilleur F1 FLAML : {best_f1}")
print(f"Meilleur modèle   : {engine_300.results['best_estimator']}")
print(f"Différence vs baseline : {'+' if diff>=0 else ''}{diff}")

EXPÉRIENCE 2 — FLAML budget étendu (300s)
Lancement FLAML — budget : 300s ...

Meilleur modèle   : lgbm
Meilleure config  : {'n_estimators': 16, 'num_leaves': 9, 'min_child_samples': 40, 'learning_rate': np.float64(0.8652439717907552), 'log_max_bin': 6, 'colsample_bytree': np.float64(0.44203150060575386), 'reg_alpha': np.float64(0.02467010390573768), 'reg_lambda': np.float64(1.6729586403348213)}
Temps total       : 300.09s

RÉSULTATS — 3 configurations comparées
Méthode                      F1         Accuracy     AUC-ROC    Temps
----------------------------------------------------------------------
Baseline RF défaut           0.8814     0.8852       N/A        0.12s
FLAML 120s (XGBoost)         0.8526     0.8525       0.9367     120s
FLAML 300s                   0.8691     0.8689       0.9361     300s
----------------------------------------------------------------------

Meilleur F1 FLAML : 0.8691
Meilleur modèle   : lgbm
Différence vs baseline : -0.0123


In [6]:
# ═══════════════════════════════════════════════════
# EXPÉRIENCE 3 — FLAML sur Diabetes (768 lignes)
# Dataset plus grand → FLAML devrait mieux performer
# ═══════════════════════════════════════════════════
print("=" * 55)
print("EXPÉRIENCE 3 — FLAML sur Diabetes")
print("Dataset : 768 lignes | Budget : 120s")
print("=" * 55)

# Preprocessing automatique
preprocessor_d = AutoPreprocessor()
X_diabetes     = preprocessor_d.fit_transform(df_diabetes.drop('target', axis=1))
y_diabetes     = df_diabetes['target']

# Baseline Diabetes
from sklearn.ensemble import RandomForestClassifier
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42, stratify=y_diabetes)

rf_d = RandomForestClassifier(n_estimators=100, random_state=42)
rf_d.fit(X_tr_d, y_tr_d)
baseline_d_f1  = round(f1_score(y_te_d, rf_d.predict(X_te_d), average='weighted'), 4)
baseline_d_acc = round(accuracy_score(y_te_d, rf_d.predict(X_te_d)), 4)
print(f"\nBaseline Diabetes — F1 : {baseline_d_f1} | Accuracy : {baseline_d_acc}")

# FLAML sur Diabetes
print("\nLancement FLAML sur Diabetes...")
engine_d = AutoMLEngine(task='classification', time_budget=120)
engine_d.fit(X_tr_d, y_tr_d)
results_d = engine_d.evaluate(X_te_d, y_te_d)

print("\n" + "=" * 55)
print("RÉSULTATS — Diabetes")
print("=" * 55)
print(f"{'Méthode':<25} {'F1':<10} {'Accuracy':<12} {'AUC-ROC'}")
print("-" * 55)
print(f"{'Baseline RF défaut':<25} {baseline_d_f1:<10} {baseline_d_acc:<12} {'N/A'}")
print(f"{'FLAML 120s':<25} {results_d['f1_score']:<10} {results_d['accuracy']:<12} {results_d['auc_roc']}")
print("-" * 55)

diff_d = round(results_d['f1_score'] - baseline_d_f1, 4)
print(f"\nAmélioration F1 : {'+' if diff_d >= 0 else ''}{diff_d}")
print(f"Meilleur modèle : {engine_d.results['best_estimator']}")
if diff_d > 0:
    print("FLAML bat la baseline sur Diabetes ✓")
else:
    print("Résultat à analyser...")

EXPÉRIENCE 3 — FLAML sur Diabetes
Dataset : 768 lignes | Budget : 120s

Baseline Diabetes — F1 : 0.7481 | Accuracy : 0.7532

Lancement FLAML sur Diabetes...
Lancement FLAML — budget : 120s ...

Meilleur modèle   : xgboost
Meilleure config  : {'n_estimators': 6, 'max_leaves': 7, 'min_child_weight': np.float64(3.0786300511775786), 'learning_rate': np.float64(0.651851180016303), 'subsample': np.float64(0.9182630894654833), 'colsample_bylevel': np.float64(0.9206736504688494), 'colsample_bytree': np.float64(0.801112830671241), 'reg_alpha': np.float64(0.0026784891535010266), 'reg_lambda': np.float64(0.43302272178918705)}
Temps total       : 120.01s

RÉSULTATS — Diabetes
Méthode                   F1         Accuracy     AUC-ROC
-------------------------------------------------------
Baseline RF défaut        0.7481     0.7532       N/A
FLAML 120s                0.7319     0.7338       0.8007
-------------------------------------------------------

Amélioration F1 : -0.0162
Meilleur modèle : x

In [7]:
# ═══════════════════════════════════════════════════
# TÉLÉCHARGEMENT DES GRANDS DATASETS
# ═══════════════════════════════════════════════════
import urllib.request

# Dataset 3 — Telco Customer Churn (7 043 lignes)
url_churn = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
urllib.request.urlretrieve(url_churn, '../datasets/raw/telco_churn.csv')
print("✓ Telco Churn téléchargé (7 043 lignes)")

# Dataset 4 — Credit Risk / Give Me Some Credit (Kaggle alternatif)
url_credit = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/german_credit_data.csv"
urllib.request.urlretrieve(url_credit, '../datasets/raw/credit_risk.csv')
print("✓ Credit Risk téléchargé")

print("\nVérification des fichiers :")
for f in ['telco_churn.csv', 'credit_risk.csv']:
    df_tmp = pd.read_csv(f'../datasets/raw/{f}')
    print(f"  {f} : {df_tmp.shape[0]} lignes × {df_tmp.shape[1]} colonnes")

✓ Telco Churn téléchargé (7 043 lignes)


HTTPError: HTTP Error 404: Not Found

In [ ]:
# ═══════════════════════════════════════════════════
# TÉLÉCHARGEMENT Credit Risk — URL corrigée
# ═══════════════════════════════════════════════════

# Credit Risk — South German Credit (1000 lignes)
url_credit = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/credit.csv"
urllib.request.urlretrieve(url_credit, '../datasets/raw/credit_risk.csv')
print("✓ Credit Risk téléchargé")

# Vérification des 2 datasets
print("\nVérification des fichiers :")
for f in ['telco_churn.csv', 'credit_risk.csv']:
    df_tmp = pd.read_csv(f'../datasets/raw/{f}')
    print(f"  {f} : {df_tmp.shape[0]} lignes × {df_tmp.shape[1]} colonnes")
    print(f"  Colonnes : {df_tmp.columns.tolist()}")
    print()

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUTION — Datasets depuis sources fiables
# ═══════════════════════════════════════════════════

# Dataset 3 — Telco Churn déjà téléchargé ✓
df_churn = pd.read_csv('../datasets/raw/telco_churn.csv')
print(f"✓ Telco Churn : {df_churn.shape[0]} lignes × {df_churn.shape[1]} colonnes")

# Dataset 4 — Credit Risk généré depuis OpenML
from sklearn.datasets import fetch_openml

print("\nTéléchargement Credit Risk depuis OpenML...")
credit = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
df_credit = credit.frame
print(f"✓ Credit Risk : {df_credit.shape[0]} lignes × {df_credit.shape[1]} colonnes")
print(f"  Colonnes : {df_credit.columns.tolist()}")
print(f"  Distribution cible : {df_credit['class'].value_counts().to_dict()}")

# Sauvegarder
df_credit.to_csv('../datasets/raw/credit_risk.csv', index=False)
print("\n✓ Credit Risk sauvegardé dans datasets/raw/")

# Résumé
print("\n" + "=" * 45)
print("DATASETS DISPONIBLES")
print("=" * 45)
datasets_info = [
    ("Heart Disease",  303,  "Santé"),
    ("Diabetes",       768,  "Santé"),
    ("Telco Churn",    df_churn.shape[0], "Télécom"),
    ("Credit Risk",    df_credit.shape[0], "Finance"),
]
for name, rows, domain in datasets_info:
    print(f"  {name:<20} {rows:>6} lignes   {domain}")

In [ ]:
# ═══════════════════════════════════════════════════
# AJOUT — Adult Income Dataset (48 842 lignes)
# Prédit si le revenu annuel > 50K$
# Très utilisé en benchmark AutoML
# ═══════════════════════════════════════════════════

print("Téléchargement Adult Income depuis OpenML...")
adult = fetch_openml(name='adult', version=2, as_frame=True, parser='auto')
df_adult = adult.frame

print(f"✓ Adult Income : {df_adult.shape[0]} lignes × {df_adult.shape[1]} colonnes")
print(f"  Distribution cible : {df_adult['class'].value_counts().to_dict()}")

# Sauvegarder
df_adult.to_csv('../datasets/raw/adult_income.csv', index=False)
print("✓ Adult Income sauvegardé")

# Résumé final
print("\n" + "=" * 50)
print("DATASETS FINAUX DISPONIBLES")
print("=" * 50)
datasets_info = [
    ("Heart Disease",  303,    "Santé",    "Petit"),
    ("Diabetes",       768,    "Santé",    "Petit"),
    ("Credit Risk",    1000,   "Finance",  "Moyen"),
    ("Telco Churn",    7043,   "Télécom",  "Grand"),
    ("Adult Income",   df_adult.shape[0], "Finance", "Très grand"),
]
print(f"{'Dataset':<20} {'Lignes':>8}   {'Domaine':<12} {'Taille'}")
print("-" * 50)
for name, rows, domain, size in datasets_info:
    print(f"{name:<20} {rows:>8}   {domain:<12} {size}")
print(f"\nTotal : {sum(r for _,r,_,_ in datasets_info):,} lignes de données !")

In [ ]:
# ═══════════════════════════════════════════════════
# AJOUT — Dataset Santé grand
# Stroke Prediction Dataset (21 000+ lignes)
# Prédit si un patient va avoir un AVC
# Très pertinent pour le contexte africain
# ═══════════════════════════════════════════════════

print("Téléchargement Stroke Prediction depuis OpenML...")
stroke = fetch_openml(name='stroke-prediction', version=1, as_frame=True, parser='auto')
df_stroke = stroke.frame
print(f"✓ Stroke : {df_stroke.shape[0]} lignes × {df_stroke.shape[1]} colonnes")
print(f"  Distribution cible : {df_stroke['stroke'].value_counts().to_dict()}")
df_stroke.to_csv('../datasets/raw/stroke.csv', index=False)
print("✓ Stroke sauvegardé")

In [ ]:
# ═══════════════════════════════════════════════════
# AJOUT — Grand dataset Santé via ID OpenML
# ═══════════════════════════════════════════════════

# Essai 1 — Heart Disease Cleveland étendu (ID 1565)
print("Tentative 1 — Heart Disease étendu...")
try:
    ds = fetch_openml(data_id=1565, as_frame=True, parser='auto')
    df_health_big = ds.frame
    print(f"✓ {ds.details['name']} : {df_health_big.shape[0]} lignes × {df_health_big.shape[1]} colonnes")
    df_health_big.to_csv('../datasets/raw/health_big.csv', index=False)
except Exception as e:
    print(f"Echec : {e}")

    # Essai 2 — Mammography (11 183 lignes) dataset santé
    print("\nTentative 2 — Mammography dataset...")
    try:
        ds2 = fetch_openml(data_id=310, as_frame=True, parser='auto')
        df_health_big = ds2.frame
        print(f"✓ {ds2.details['name']} : {df_health_big.shape[0]} lignes × {df_health_big.shape[1]} colonnes")
        df_health_big.to_csv('../datasets/raw/health_big.csv', index=False)
    except Exception as e2:
        print(f"Echec : {e2}")

        # Essai 3 — Diabetes 130 hospitals (101 766 lignes)
        print("\nTentative 3 — Diabetes 130 Hospitals (grand)...")
        ds3 = fetch_openml(data_id=4541, as_frame=True, parser='auto')
        df_health_big = ds3.frame
        print(f"✓ {ds3.details['name']} : {df_health_big.shape[0]} lignes × {df_health_big.shape[1]} colonnes")
        df_health_big.to_csv('../datasets/raw/health_big.csv', index=False)

print("\nDataset santé grand sauvegardé ✓")
print(f"Shape finale : {df_health_big.shape}")

In [ ]:
# ═══════════════════════════════════════════════════
# GRAND dataset Santé — Diabetes 130 Hospitals
# 101 766 lignes — données hospitalières réelles
# ═══════════════════════════════════════════════════

print("Téléchargement Diabetes 130 Hospitals (100K+ lignes)...")
try:
    ds_big = fetch_openml(data_id=4541, as_frame=True, parser='auto')
    df_health_big = ds_big.frame
    print(f"✓ {ds_big.details['name']}")
    print(f"  Lignes   : {df_health_big.shape[0]:,}")
    print(f"  Colonnes : {df_health_big.shape[1]}")
    df_health_big.to_csv('../datasets/raw/health_big.csv', index=False)
    print("✓ Sauvegardé !")
except Exception as e:
    print(f"Echec ID 4541 : {e}")
    
    # Alternative — Skin Segmentation (245 057 lignes)
    print("\nAlternative — Skin Segmentation Dataset...")
    ds_skin = fetch_openml(data_id=1502, as_frame=True, parser='auto')
    df_health_big = ds_skin.frame
    print(f"✓ {ds_skin.details['name']}")
    print(f"  Lignes   : {df_health_big.shape[0]:,}")
    df_health_big.to_csv('../datasets/raw/health_big.csv', index=False)

# Résumé final de tous les datasets
print("\n" + "=" * 55)
print("TOUS LES DATASETS DU PROJET")
print("=" * 55)
print(f"{'Dataset':<22} {'Lignes':>10}   {'Domaine':<12}")
print("-" * 55)
all_datasets = [
    ("Heart Disease",    303,    "Santé"),
    ("Diabetes",         768,    "Santé"),
    ("Credit Risk",      1000,   "Finance"),
    ("Telco Churn",      7043,   "Télécom"),
    ("Adult Income",     48842,  "Finance"),
    ("Health Big",       df_health_big.shape[0], "Santé"),
]
for name, rows, domain in all_datasets:
    print(f"{name:<22} {rows:>10,}   {domain}")
total = sum(r for _,r,_ in all_datasets)
print(f"\n{'TOTAL':<22} {total:>10,}   lignes")

In [ ]:
# ═══════════════════════════════════════════════════
# NETTOYAGE ET PRÉPARATION DES 4 DATASETS
# ═══════════════════════════════════════════════════

print("NETTOYAGE DES 4 DATASETS")
print("=" * 55)

# ── DATASET 1 : Heart Disease ──────────────────────
print("\n1. Heart Disease...")
df_hd = pd.read_csv('../datasets/processed/heart_disease_clean.csv')
# Déjà nettoyé en semaine 2 — juste vérifier
print(f"   Lignes       : {df_hd.shape[0]}")
print(f"   Manquantes   : {df_hd.isnull().sum().sum()}")
print(f"   Cible        : {df_hd['target'].value_counts().to_dict()}")
df_hd.to_csv('../datasets/processed/ds1_heart_disease.csv', index=False)
print("   ✓ Sauvegardé → ds1_heart_disease.csv")

# ── DATASET 2 : Telco Churn ────────────────────────
print("\n2. Telco Churn...")
df_ch = pd.read_csv('../datasets/raw/telco_churn.csv')

# Supprimer colonne inutile
df_ch = df_ch.drop('customerID', axis=1)

# Convertir TotalCharges en numérique
df_ch['TotalCharges'] = pd.to_numeric(df_ch['TotalCharges'], errors='coerce')

# Convertir cible en binaire
df_ch['target'] = (df_ch['Churn'] == 'Yes').astype(int)
df_ch = df_ch.drop('Churn', axis=1)

# Encoder les colonnes Yes/No en binaire
yes_no_cols = ['Partner','Dependents','PhoneService','PaperlessBilling',
               'MultipleLines','OnlineSecurity','OnlineBackup',
               'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
for col in yes_no_cols:
    if col in df_ch.columns:
        df_ch[col] = df_ch[col].map({'Yes':1,'No':0,'No phone service':0,'No internet service':0})

print(f"   Lignes       : {df_ch.shape[0]}")
print(f"   Colonnes     : {df_ch.shape[1]}")
print(f"   Manquantes   : {df_ch.isnull().sum().sum()}")
print(f"   Cible        : {df_ch['target'].value_counts().to_dict()}")
df_ch.to_csv('../datasets/processed/ds2_telco_churn.csv', index=False)
print("   ✓ Sauvegardé → ds2_telco_churn.csv")

# ── DATASET 3 : Adult Income ───────────────────────
print("\n3. Adult Income...")
df_ad = pd.read_csv('../datasets/raw/adult_income.csv')

# Convertir cible en binaire
target_col = 'class'
df_ad['target'] = (df_ad[target_col].astype(str).str.strip().isin(['>50K','>50K.'])).astype(int)
df_ad = df_ad.drop(target_col, axis=1)

# Supprimer les lignes avec valeurs manquantes
df_ad = df_ad.replace('?', np.nan)
df_ad = df_ad.dropna()
df_ad = df_ad.reset_index(drop=True)

print(f"   Lignes       : {df_ad.shape[0]}")
print(f"   Colonnes     : {df_ad.shape[1]}")
print(f"   Manquantes   : {df_ad.isnull().sum().sum()}")
print(f"   Cible        : {df_ad['target'].value_counts().to_dict()}")
df_ad.to_csv('../datasets/processed/ds3_adult_income.csv', index=False)
print("   ✓ Sauvegardé → ds3_adult_income.csv")

# ── DATASET 4 : Health Big ─────────────────────────
print("\n4. Health Big (Diabetes 130 Hospitals)...")
df_hb = pd.read_csv('../datasets/raw/health_big.csv')

# Identifier la colonne cible
print(f"   Colonnes disponibles : {df_hb.columns.tolist()[:10]}...")

# La cible est 'readmitted' — on la convertit en binaire
# 0 = pas de réadmission, 1 = réadmission
if 'readmitted' in df_hb.columns:
    df_hb['target'] = (df_hb['readmitted'] != 'NO').astype(int)
    df_hb = df_hb.drop('readmitted', axis=1)
elif 'class' in df_hb.columns:
    df_hb['target'] = (df_hb['class'].astype(str) != '0').astype(int)
    df_hb = df_hb.drop('class', axis=1)

# Supprimer colonnes avec trop de valeurs manquantes (>50%)
threshold = len(df_hb) * 0.5
df_hb = df_hb.dropna(thresh=threshold, axis=1)

# Supprimer colonnes identifiants inutiles
id_cols = [c for c in df_hb.columns if 'id' in c.lower() or 'ID' in c]
df_hb = df_hb.drop(columns=id_cols, errors='ignore')

# Remplacer '?' par NaN
df_hb = df_hb.replace('?', np.nan)

print(f"   Lignes       : {df_hb.shape[0]}")
print(f"   Colonnes     : {df_hb.shape[1]}")
print(f"   Manquantes   : {df_hb.isnull().sum().sum()}")
print(f"   Cible        : {df_hb['target'].value_counts().to_dict()}")
df_hb.to_csv('../datasets/processed/ds4_health_big.csv', index=False)
print("   ✓ Sauvegardé → ds4_health_big.csv")

# ── RÉSUMÉ FINAL ───────────────────────────────────
print("\n" + "=" * 55)
print("RÉSUMÉ — DATASETS PRÊTS POUR EXPÉRIENCES")
print("=" * 55)
print(f"{'#':<4} {'Dataset':<22} {'Lignes':>8}  {'Colonnes':>9}  {'Domaine'}")
print("-" * 55)
datasets_ready = [
    ("1", "Heart Disease",  df_hd.shape[0],  df_hd.shape[1],  "Santé"),
    ("2", "Telco Churn",    df_ch.shape[0],  df_ch.shape[1],  "Télécom"),
    ("3", "Adult Income",   df_ad.shape[0],  df_ad.shape[1],  "Finance"),
    ("4", "Health Big",     df_hb.shape[0],  df_hb.shape[1],  "Santé"),
]
for num, name, rows, cols, domain in datasets_ready:
    print(f"{num:<4} {name:<22} {rows:>8,}  {cols:>9}  {domain}")
print(f"\n{'TOTAL':<26} {sum(r for _,_,r,_,_ in datasets_ready):>8,}")
print("\nTous les datasets sont prêts pour les expériences ! ✓")

In [ ]:
# ═══════════════════════════════════════════════════
# EXPÉRIENCES FLAML — 4 DATASETS
# ═══════════════════════════════════════════════════
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

DATASETS = [
    {'name':'Heart Disease', 'file':'../datasets/processed/ds1_heart_disease.csv',
     'target':'target', 'domaine':'Sante', 'lignes':303},
    {'name':'Telco Churn',   'file':'../datasets/processed/ds2_telco_churn.csv',
     'target':'target', 'domaine':'Telecom', 'lignes':7043},
    {'name':'Adult Income',  'file':'../datasets/processed/ds3_adult_income.csv',
     'target':'target', 'domaine':'Finance', 'lignes':45222},
    {'name':'Health Big',    'file':'../datasets/processed/ds4_health_big.csv',
     'target':'target', 'domaine':'Sante', 'lignes':101766},
]

all_results = []

for ds in DATASETS:
    print(f"\n{'='*55}")
    print(f"Dataset : {ds['name']} ({ds['lignes']:,} lignes)")
    print(f"{'='*55}")

    df = pd.read_csv(ds['file'])
    X  = df.drop(ds['target'], axis=1)
    y  = df[ds['target']]

    prep   = AutoPreprocessor()
    X_proc = prep.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_proc, y, test_size=0.2, random_state=42, stratify=y)

    # Baseline
    print("  Baseline RF...")
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    t0 = time.time()
    rf.fit(X_train, y_train)
    t_rf   = round(time.time() - t0, 2)
    bl_f1  = round(f1_score(y_test, rf.predict(X_test), average='weighted'), 4)
    bl_acc = round(accuracy_score(y_test, rf.predict(X_test)), 4)
    print(f"  Baseline  -> F1: {bl_f1} | Acc: {bl_acc} | Temps: {t_rf}s")

    # FLAML
    print("  FLAML 120s...")
    engine = AutoMLEngine(task='classification', time_budget=120)
    engine.fit(X_train, y_train)
    res = engine.evaluate(X_test, y_test)
    print(f"  FLAML     -> F1: {res['f1_score']} | Acc: {res['accuracy']} | Modele: {res['best_estimator']}")

    diff = round(res['f1_score'] - bl_f1, 4)
    gain_str = f"+{diff}" if diff >= 0 else str(diff)
    winner = "FLAML GAGNE" if diff > 0 else "A ANALYSER"
    print(f"  Gain F1   -> {gain_str}  [{winner}]")

    all_results.append({
        'dataset'     : ds['name'],
        'domaine'     : ds['domaine'],
        'lignes'      : ds['lignes'],
        'baseline_f1' : bl_f1,
        'baseline_acc': bl_acc,
        'flaml_f1'    : res['f1_score'],
        'flaml_acc'   : res['accuracy'],
        'flaml_auc'   : res['auc_roc'],
        'flaml_model' : res['best_estimator'],
        'flaml_time'  : res['training_time'],
        'gain_f1'     : diff,
    })

# Tableau final
print(f"\n{'='*65}")
print("TABLEAU COMPARATIF FINAL — FLAML vs BASELINE")
print(f"{'='*65}")
print(f"{'Dataset':<20} {'Lignes':>8}  {'BL_F1':>6}  {'FL_F1':>6}  {'Gain':>7}  Modele")
print("-"*65)
for r in all_results:
    g = f"+{r['gain_f1']}" if r['gain_f1'] >= 0 else str(r['gain_f1'])
    print(f"{r['dataset']:<20} {r['lignes']:>8,}  {r['baseline_f1']:>6}  {r['flaml_f1']:>6}  {g:>7}  {r['flaml_model']}")
print(f"{'='*65}")

import json
with open('../reports/flaml_results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)
print("\nResultats sauvegardes dans reports/flaml_results.json OK")

In [ ]:
# ═══════════════════════════════════════════════════
# AUTOPREPROCESSOR CORRIGÉ — encodage arbitrary
# ═══════════════════════════════════════════════════
from sklearn.impute import SimpleImputer
from feature_engine.encoding import OrdinalEncoder
from feature_engine.outliers import Winsorizer

class AutoPreprocessor:
    def __init__(self):
        self.num_cols        = []
        self.cat_cols        = []
        self.binary_cols     = []
        self.continuous_cols = []
        self.fitted          = False

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        self.num_cols        = X.select_dtypes(include=['number']).columns.tolist()
        self.cat_cols        = X.select_dtypes(include=['object','category']).columns.tolist()
        self.binary_cols     = [c for c in self.num_cols if X[c].nunique() <= 2]
        self.continuous_cols = [c for c in self.num_cols if X[c].nunique() > 2]

        # 1. Imputation valeurs manquantes
        if self.num_cols:
            imputer = SimpleImputer(strategy='median')
            X[self.num_cols] = imputer.fit_transform(X[self.num_cols])

        # 2. Imputation catégorielle
        if self.cat_cols:
            for col in self.cat_cols:
                X[col] = X[col].fillna(X[col].mode()[0])

        # 3. Encodage catégoriel — arbitrary (pas besoin de y)
        if self.cat_cols:
            encoder = OrdinalEncoder(
                encoding_method='arbitrary',
                variables=self.cat_cols
            )
            X = encoder.fit_transform(X)

        # 4. Outliers sur colonnes continues uniquement
        if self.continuous_cols:
            try:
                winsorizer = Winsorizer(
                    variables=self.continuous_cols,
                    capping_method='iqr',
                    tail='both', fold=1.5
                )
                X = winsorizer.fit_transform(X)
            except Exception as e:
                print(f"    Winsorizer ignore : {e}")

        self.fitted = True
        return X

    def get_info(self):
        return {
            "colonnes_numeriques"    : self.num_cols,
            "colonnes_binaires"      : self.binary_cols,
            "colonnes_continues"     : self.continuous_cols,
            "colonnes_categorielles" : self.cat_cols,
        }

print("AutoPreprocessor v2 defini OK")
print("Correction : encodage arbitrary — fonctionne sans y")

In [ ]:
# ═══════════════════════════════════════════════════
# RELANCE EXPÉRIENCES — tous les datasets
# (repart de Telco Churn — Heart Disease déjà fait)
# ═══════════════════════════════════════════════════

# On garde le résultat Heart Disease déjà calculé
all_results = [
    {
        'dataset'     : 'Heart Disease',
        'domaine'     : 'Sante',
        'lignes'      : 303,
        'baseline_f1' : 0.9017,
        'baseline_acc': 0.9016,
        'flaml_f1'    : 0.8526,
        'flaml_acc'   : 0.8525,
        'flaml_auc'   : 0.9367,
        'flaml_model' : 'xgboost',
        'flaml_time'  : 119.99,
        'gain_f1'     : -0.0491,
    }
]

# Lancer sur les 3 datasets restants
DATASETS_REMAINING = [
    {'name':'Telco Churn',  'file':'../datasets/processed/ds2_telco_churn.csv',
     'target':'target', 'domaine':'Telecom', 'lignes':7043},
    {'name':'Adult Income', 'file':'../datasets/processed/ds3_adult_income.csv',
     'target':'target', 'domaine':'Finance', 'lignes':45222},
    {'name':'Health Big',   'file':'../datasets/processed/ds4_health_big.csv',
     'target':'target', 'domaine':'Sante', 'lignes':101766},
]

for ds in DATASETS_REMAINING:
    print(f"\n{'='*55}")
    print(f"Dataset : {ds['name']} ({ds['lignes']:,} lignes)")
    print(f"{'='*55}")

    df     = pd.read_csv(ds['file'])
    X      = df.drop(ds['target'], axis=1)
    y      = df[ds['target']]

    prep   = AutoPreprocessor()
    X_proc = prep.fit_transform(X)
    print(f"  Preprocessing OK — {X_proc.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X_proc, y, test_size=0.2, random_state=42, stratify=y)

    # Baseline
    print("  Baseline RF...")
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    t0 = time.time()
    rf.fit(X_train, y_train)
    t_rf   = round(time.time() - t0, 2)
    bl_f1  = round(f1_score(y_test, rf.predict(X_test), average='weighted'), 4)
    bl_acc = round(accuracy_score(y_test, rf.predict(X_test)), 4)
    print(f"  Baseline  -> F1: {bl_f1} | Acc: {bl_acc} | Temps: {t_rf}s")

    # FLAML
    print("  FLAML 120s...")
    engine = AutoMLEngine(task='classification', time_budget=120)
    engine.fit(X_train, y_train)
    res    = engine.evaluate(X_test, y_test)
    diff   = round(res['f1_score'] - bl_f1, 4)
    g      = f"+{diff}" if diff >= 0 else str(diff)
    winner = "FLAML GAGNE" if diff > 0 else "A ANALYSER"
    print(f"  FLAML     -> F1: {res['f1_score']} | Modele: {res['best_estimator']}")
    print(f"  Gain F1   -> {g}  [{winner}]")

    all_results.append({
        'dataset'     : ds['name'],
        'domaine'     : ds['domaine'],
        'lignes'      : ds['lignes'],
        'baseline_f1' : bl_f1,
        'baseline_acc': bl_acc,
        'flaml_f1'    : res['f1_score'],
        'flaml_acc'   : res['accuracy'],
        'flaml_auc'   : res['auc_roc'],
        'flaml_model' : res['best_estimator'],
        'flaml_time'  : res['training_time'],
        'gain_f1'     : diff,
    })

# Tableau final
print(f"\n{'='*65}")
print("TABLEAU COMPARATIF FINAL — FLAML vs BASELINE")
print(f"{'='*65}")
print(f"{'Dataset':<20} {'Lignes':>8}  {'BL_F1':>6}  {'FL_F1':>6}  {'Gain':>7}  Modele")
print("-"*65)
for r in all_results:
    g = f"+{r['gain_f1']}" if r['gain_f1'] >= 0 else str(r['gain_f1'])
    print(f"{r['dataset']:<20} {r['lignes']:>8,}  {r['baseline_f1']:>6}  {r['flaml_f1']:>6}  {g:>7}  {r['flaml_model']}")
print(f"{'='*65}")

import json
with open('../reports/flaml_results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)
print("\nResultats sauvegardes OK")

In [ ]:
# ═══════════════════════════════════════════════════
# EXPÉRIENCES OPTUNA — 4 DATASETS
# Comparaison avec FLAML
# ═══════════════════════════════════════════════════
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import xgboost as xgb
from sklearn.model_selection import cross_val_score

def run_optuna(X_train, y_train, X_test, y_test, timeout=120):
    def objective(trial):
        params = {
            'n_estimators'     : trial.suggest_int('n_estimators', 50, 500),
            'max_depth'        : trial.suggest_int('max_depth', 3, 10),
            'learning_rate'    : trial.suggest_float('lr', 1e-4, 0.3, log=True),
            'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        }
        model = xgb.XGBClassifier(
            **params, eval_metric='logloss',
            verbosity=0, random_state=42)
        score = cross_val_score(
            model, X_train, y_train,
            cv=3, scoring='f1_weighted').mean()
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, timeout=timeout, show_progress_bar=False)

    # Entraîner le meilleur modèle
    best = study.best_params
    best_model = xgb.XGBClassifier(
        n_estimators=best['n_estimators'],
        max_depth=best['max_depth'],
        learning_rate=best['lr'],
        subsample=best['subsample'],
        colsample_bytree=best['colsample_bytree'],
        reg_alpha=best['reg_alpha'],
        reg_lambda=best['reg_lambda'],
        eval_metric='logloss', verbosity=0, random_state=42)
    best_model.fit(X_train, y_train)

    f1  = round(f1_score(y_test, best_model.predict(X_test), average='weighted'), 4)
    acc = round(accuracy_score(y_test, best_model.predict(X_test)), 4)
    n_trials = len(study.trials)
    return f1, acc, n_trials

# Lancer Optuna sur les 4 datasets
optuna_results = []
DATASETS_ALL = [
    {'name':'Heart Disease', 'file':'../datasets/processed/ds1_heart_disease.csv',
     'lignes':303},
    {'name':'Telco Churn',   'file':'../datasets/processed/ds2_telco_churn.csv',
     'lignes':7043},
    {'name':'Adult Income',  'file':'../datasets/processed/ds3_adult_income.csv',
     'lignes':45222},
    {'name':'Health Big',    'file':'../datasets/processed/ds4_health_big.csv',
     'lignes':101766},
]

for ds in DATASETS_ALL:
    print(f"\nOptuna sur {ds['name']} ({ds['lignes']:,} lignes)...")
    df     = pd.read_csv(ds['file'])
    X      = df.drop('target', axis=1)
    y      = df['target']
    prep   = AutoPreprocessor()
    X_proc = prep.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_proc, y, test_size=0.2, random_state=42, stratify=y)

    t0 = time.time()
    f1, acc, n_trials = run_optuna(X_train, y_train, X_test, y_test, timeout=120)
    t_total = round(time.time() - t0, 2)

    print(f"  Optuna -> F1: {f1} | Acc: {acc} | Trials: {n_trials} | Temps: {t_total}s")
    optuna_results.append({
        'dataset'  : ds['name'],
        'lignes'   : ds['lignes'],
        'optuna_f1': f1,
        'optuna_acc': acc,
        'n_trials' : n_trials,
        'time'     : t_total,
    })

# Tableau comparatif FINAL complet
print(f"\n{'='*75}")
print("TABLEAU FINAL — Baseline vs FLAML vs Optuna")
print(f"{'='*75}")
print(f"{'Dataset':<20} {'Lignes':>8}  {'Baseline':>9}  {'FLAML':>7}  {'Optuna':>7}  {'Gagnant'}")
print("-"*75)

flaml_map = {r['dataset']: r for r in all_results}
for i, opt in enumerate(optuna_results):
    fl  = flaml_map[opt['dataset']]
    scores = {
        'Baseline': fl['baseline_f1'],
        'FLAML'   : fl['flaml_f1'],
        'Optuna'  : opt['optuna_f1']
    }
    winner = max(scores, key=scores.get)
    print(f"{opt['dataset']:<20} {opt['lignes']:>8,}  "
          f"{fl['baseline_f1']:>9}  {fl['flaml_f1']:>7}  "
          f"{opt['optuna_f1']:>7}  {winner}")
print(f"{'='*75}")

# Sauvegarder
with open('../reports/optuna_results.json', 'w', encoding='utf-8') as f:
    json.dump(optuna_results, f, indent=4, ensure_ascii=False)
print("\nResultats Optuna sauvegardes OK")

In [ ]:
# ═══════════════════════════════════════════════════
# EXPLICATIONS SHAP — sur Telco Churn
# Dataset le plus pertinent pour les PME
# ═══════════════════════════════════════════════════
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print("Generation des explications SHAP...")
print("Dataset : Telco Churn (le plus pertinent PME)")

# Recharger et préparer Telco Churn
df_ch  = pd.read_csv('../datasets/processed/ds2_telco_churn.csv')
X_ch   = df_ch.drop('target', axis=1)
y_ch   = df_ch['target']

prep_ch   = AutoPreprocessor()
X_ch_proc = prep_ch.fit_transform(X_ch)

X_train_ch, X_test_ch, y_train_ch, y_test_ch = train_test_split(
    X_ch_proc, y_ch, test_size=0.2, random_state=42, stratify=y_ch)

# Réentraîner le meilleur modèle FLAML
engine_shap = AutoMLEngine(task='classification', time_budget=60)
engine_shap.fit(X_train_ch, y_train_ch)
best_model  = engine_shap.automl.model.estimator

print(f"\nModele utilise : {engine_shap.results['best_estimator']}")

# Générer les explications SHAP
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_ch)

# Importance globale des features
importance = pd.DataFrame({
    'feature'   : X_test_ch.columns.tolist(),
    'importance': abs(shap_values).mean(axis=0).tolist()
}).sort_values('importance', ascending=False)

print("\nTop 10 features les plus importantes (SHAP) :")
print(importance.head(10).to_string(index=False))

# Sauvegarder summary plot
shap.summary_plot(shap_values, X_test_ch, show=False, plot_size=(12,7))
plt.title("SHAP Summary Plot — Telco Churn")
plt.tight_layout()
plt.savefig('../reports/shap_telco_churn.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nGraphique SHAP sauvegarde -> reports/shap_telco_churn.png")

# Sauvegarder les importances
importance.to_csv('../reports/shap_importance_telco.csv', index=False)
print("Importances sauvegardees -> reports/shap_importance_telco.csv")

In [ ]:
# ═══════════════════════════════════════════════════
# TRACKING MLFLOW — Enregistrer toutes les expériences
# ═══════════════════════════════════════════════════
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

print("Connexion a MLflow sur http://localhost:5000")

# Enregistrer toutes les expériences FLAML
mlflow.set_experiment("AutoML_FLAML_vs_Baseline")

for r in all_results:
    with mlflow.start_run(run_name=f"FLAML_{r['dataset'].replace(' ','_')}"):
        # Paramètres
        mlflow.log_param("dataset",      r['dataset'])
        mlflow.log_param("domaine",      r['domaine'])
        mlflow.log_param("lignes",       r['lignes'])
        mlflow.log_param("methode",      "FLAML")
        mlflow.log_param("time_budget",  120)
        mlflow.log_param("best_model",   r['flaml_model'])

        # Métriques FLAML
        mlflow.log_metric("flaml_f1",       r['flaml_f1'])
        mlflow.log_metric("flaml_accuracy", r['flaml_acc'])
        mlflow.log_metric("baseline_f1",    r['baseline_f1'])
        mlflow.log_metric("gain_f1",        r['gain_f1'])
        if r['flaml_auc']:
            mlflow.log_metric("flaml_auc_roc", r['flaml_auc'])

        print(f"  Run FLAML loggue : {r['dataset']}")

# Enregistrer les expériences Optuna
mlflow.set_experiment("AutoML_Optuna_vs_Baseline")

for opt in optuna_results:
    fl = flaml_map[opt['dataset']]
    with mlflow.start_run(run_name=f"Optuna_{opt['dataset'].replace(' ','_')}"):
        mlflow.log_param("dataset",     opt['dataset'])
        mlflow.log_param("methode",     "Optuna+XGBoost")
        mlflow.log_param("lignes",      opt['lignes'])
        mlflow.log_param("n_trials",    opt['n_trials'])
        mlflow.log_param("time_budget", 120)

        mlflow.log_metric("optuna_f1",       opt['optuna_f1'])
        mlflow.log_metric("optuna_accuracy", opt['optuna_acc'])
        mlflow.log_metric("baseline_f1",     fl['baseline_f1'])
        mlflow.log_metric("gain_vs_baseline",
                          round(opt['optuna_f1'] - fl['baseline_f1'], 4))

        print(f"  Run Optuna loggue : {opt['dataset']}")

# Logguer le graphique SHAP
mlflow.set_experiment("AutoML_SHAP_Explainability")
with mlflow.start_run(run_name="SHAP_Telco_Churn"):
    mlflow.log_param("dataset", "Telco Churn")
    mlflow.log_param("methode", "FLAML+CatBoost")
    mlflow.log_metric("top_feature_importance", 0.596057)
    mlflow.log_artifact("../reports/shap_telco_churn.png")
    mlflow.log_artifact("../reports/shap_importance_telco.csv")
    print("  SHAP artifacts logues")

print("\nTous les runs sont visibles sur http://localhost:5000")
print("Ouvre MLflow dans ton navigateur pour voir les resultats !")